# Comment note-t-on un film ? Exploitation de la base de données MovieLens 32M

## Introduction et problématique

# I - MovieLens 32M : téléchargement et aperçu des données

In [16]:
import os
import zipfile

import requests
import pandas as pd

URL = "https://files.grouplens.org/datasets/movielens/ml-32m.zip"
NOM_ZIP = "ml-32m.zip"
DOSSIER = "data"

In [17]:
def telecharger(url, nom_fichier):
    """Télécharge le fichier à l'adresse url (par morceaux, car il est lourd)."""
    if os.path.exists(nom_fichier):
        print("Le ZIP existe déjà, pas de nouveau téléchargement.")
        return
    reponse = requests.get(url, stream=True)
    reponse.raise_for_status()
    with open(nom_fichier, "wb") as f:
        for morceau in reponse.iter_content(chunk_size=1024 * 1024):
            f.write(morceau)
    print("Téléchargement terminé.")


def extraire(nom_zip, dossier):
    """Dézippe l'archive dans le dossier indiqué."""
    with zipfile.ZipFile(nom_zip) as archive:
        print("Contenu du ZIP :", archive.namelist())
        archive.extractall(dossier)

In [18]:
telecharger(URL, NOM_ZIP)
extraire(NOM_ZIP, DOSSIER)

Le ZIP existe déjà, pas de nouveau téléchargement.
Contenu du ZIP : ['ml-32m/', 'ml-32m/tags.csv', 'ml-32m/links.csv', 'ml-32m/README.txt', 'ml-32m/checksums.txt', 'ml-32m/ratings.csv', 'ml-32m/movies.csv']


In [19]:
chemin = os.path.join(DOSSIER, "ml-32m")

movies = pd.read_csv(os.path.join(chemin, "movies.csv"))
tags = pd.read_csv(os.path.join(chemin, "tags.csv"))
links = pd.read_csv(os.path.join(chemin, "links.csv"))
ratings = pd.read_csv(os.path.join(chemin, "ratings.csv"))

On affiche le premier dataframe, qui comporte une clé d'identification des films (movieId), le titre ainsi que leur année, et le genre associé. 
La base de données comporte 87585 films.

In [36]:
print(movies.shape)
movies.head()

(87585, 3)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


On affiche aussi les notes de ces films, qui vont être notre base de donnée principale. Le fichier est particulièrement volumineux (32 millions de notes). L'userId permet de voir quel utilisateur a mis quelle note, ce qui sera utile. 

In [21]:
print(ratings.shape)
ratings.head()

(32000204, 4)


,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858


La base de données donne aussi accès à des tags, c'est-à-dire des mots que les utilisateurs ont associé aux films. Nous choisissons de ne pas les étudier dans le cadre de ce projet.

In [22]:
print(tags.shape)
tags.head()

(2000072, 4)


,userId,movieId,tag,timestamp
0,22,26479,Kevin Kline,1583038886
1,22,79592,misogyny,1581476297
2,22,247150,acrophobia,1622483469
3,34,2174,music,1249808064
4,34,2174,weird,1249808102


Ce dataframe permet de faire des jointures avec des bases de données extérieures, en donnant la clé correspondante pour les films dans cette base et dans les bases IMDB et TMDB.

In [23]:
print(links.shape)
links.head()

(87585, 3)


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


## Création de dataframes utilisables pour l'analyse

On peut d'abord séparer l'année et le titre du film dans le dataframe movies.

In [ ]:
movies_with_date = movies.copy()

# 1. on récupère l'année (4 chiffres entre parenthèses, à la fin du titre)
movies_with_date["year"] = movies_with_date["title"].str.extract(r"\((\d{4})\)\s*$", expand=False)

# 2. on enlève l'année du titre
movies_with_date["title"] = movies_with_date["title"].str.replace(r"\s*\(\d{4}\)\s*$", "", regex=True)

# 3. on convertit l'année en nombre
movies_with_date["year"] = pd.to_numeric(movies_with_date["year"])

movies_with_date

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995.0
1,2,Jumanji,Adventure|Children|Fantasy,1995.0
2,3,Grumpier Old Men,Comedy|Romance,1995.0
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995.0
4,5,Father of the Bride Part II,Comedy,1995.0


Certains films n'ont pas de date, 617 sur 80 000 : 

In [30]:
movies_with_date[movies_with_date["year"].isna()]

,movieId,title,genres,year
15038,79607,"Millions Game, The (Das Millionenspiel)",Action|Drama|Sci-Fi|Thriller,NaN
18796,98063,Mona and the Time of Burning Love (Mona ja pal...,Drama,NaN
25425,123619,Terrible Joe Moran,(no genres listed),NaN
26356,125571,The Court-Martial of Jackie Robinson,(no genres listed),NaN
26381,125632,In Our Garden,(no genres listed),NaN
...,...,...,...,...
86528,288927,Fortunes,Comedy,NaN
86634,289193,The Wonderful Story of Henry Sugar,Adventure|Children|Comedy,NaN
86833,289909,Rattler,(no genres listed),NaN
87074,290775,Being Romanian: A Family Journal,(no genres listed),NaN


On exclut les films sans années, et on vérifie qu'elles ont bien été exclues. On réindexe bien les lignes aussi, sinon il y a un écart de 617 entre le nombre de lignes et le plus grand index. On peut aussi transformer les flottants en entiers maintenant qu'il n'y a plus de Na dans la colonne year.

In [41]:
movies_with_date = movies_with_date.dropna(subset=["year"]).reset_index(drop=True)
movies_with_date["year"].isna().sum()
movies_with_date["year"] = movies_with_date["year"].astype(int)


In [42]:
movies_with_date

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995
...,...,...,...,...
86963,292731,The Monroy Affaire,Drama,2022
86964,292737,Shelter in Solitude,Comedy|Drama,2023
86965,292753,Orca,Drama,2023
86966,292755,The Angry Breed,Drama,1968


On peut aussi faire une jointure des notes et des films pour avoir le nom du film à côté de sa note. 
On commence par rajouter une colonne qui convertit le timestamp en heure lisible. Dans MovieLens, timestamp est un nombre de secondes écoulées depuis le 1er janvier 1970 (temps Unix).

In [47]:
ratings["date"] = pd.to_datetime(ratings["timestamp"], unit="s")
ratings.head()

,userId,movieId,rating,timestamp,date
0,1,17,4.0,944249077,1999-12-03 19:24:37
1,1,25,1.0,944250228,1999-12-03 19:43:48
2,1,29,2.0,943230976,1999-11-22 00:36:16
3,1,30,5.0,944249077,1999-12-03 19:24:37
4,1,32,5.0,943228858,1999-11-22 00:00:58


In [48]:
ratings_movies = pd.merge(ratings, movies_with_date, on='movieId', how='inner')

In [49]:
ratings_movies

,userId,movieId,rating,timestamp,date,title,genres,year
0,1,17,4.0,944249077,1999-12-03 19:24:37,Sense and Sensibility,Drama|Romance,1995
1,1,25,1.0,944250228,1999-12-03 19:43:48,Leaving Las Vegas,Drama|Romance,1995
2,1,29,2.0,943230976,1999-11-22 00:36:16,"City of Lost Children, The (Cité des enfants p...",Adventure|Drama|Fantasy|Mystery|Sci-Fi,1995
3,1,30,5.0,944249077,1999-12-03 19:24:37,Shanghai Triad (Yao a yao yao dao waipo qiao),Crime|Drama,1995
4,1,32,5.0,943228858,1999-11-22 00:00:58,Twelve Monkeys (a.k.a. 12 Monkeys),Mystery|Sci-Fi|Thriller,1995
...,...,...,...,...,...,...,...,...
31965033,200948,79702,4.5,1294412589,2011-01-07 15:03:09,Scott Pilgrim vs. the World,Action|Comedy|Fantasy|Musical|Romance,2010
31965034,200948,79796,1.0,1287216292,2010-10-16 08:04:52,Centurion,Action|Adventure|Drama|Thriller|War,2010
31965035,200948,80350,0.5,1294412671,2011-01-07 15:04:31,Vampires Suck,Comedy,2010
31965036,200948,80463,3.5,1350423800,2012-10-16 21:43:20,"Social Network, The",Drama,2010


# III - Première approche du dataset

statistiques descriptives : quand sont sortis les films (histogramme) => peut-être pour Gabriel